# 原 Stage1 尾部概率条件组成：固定 0bp 账户

假设：原有 tail head 对 SOXX 未来五开盘路径跌幅的概率在尾部风险高时应减少 SOXL 杠杆与现金回收，在风险低时沿用已固定的名义 beta 组成。概率连续进入目标，不设后验阈值。完整 Stage1 分配、动作、风控、恢复及 Tiered 账户链为父方案；SOXX 为最终硬门。

In [1]:
import os,json,hashlib,tempfile,subprocess,shlex,tarfile
from pathlib import Path
import numpy as np,pandas as pd,nbformat
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
from qlib.workflow import R
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import _settings,archive_nas_recorder
from src.backtest.native_raw_corporate import native_raw_validate
from src.backtest.stage1_domain_heads_raw import replay_stage1_raw
from src.model.tail_probability_composition import AuditedTailProbabilityComposition
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.stage1_nvda_guidance_receiver import prepare_account_sources
from src.workflow.stage1_rolling126_shortfall import _analyze_account
root=Path.cwd().resolve();card=root/'research/stage1_tail_probability_composition_0bp_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal'] and not (card/'run_state.json').exists()
assert hashlib.sha256(nbformat.read(card/'stage1_tail_probability_composition_0bp_20261023.ipynb',as_version=4).cells[1].source.encode()).hexdigest()==plan['code_cell_sha256']
for name,digest in plan['identities'].items():assert sha(root/name)==digest,name
helper=root/plan['account_source_helper'];before=sha(helper/'account_source_preflight.json');assert before==plan['account_source_receipt_sha256']
account=prepare_account_sources(root=root,card=helper)
assert sha(helper/'account_source_preflight.json')==before
assert len(account['frame'])==944 and len(account['points'])==3778
assert all(account['references'][role]['audit']['status']=='passed' for role in ('Stage1_A','SOXX'))
provider=root/plan['provider_uri'];assert sha(provider/'qlib_store_manifest.json')==plan['provider_manifest_sha256']
frame=account['frame'].copy(deep=True);assert len(frame)==944 and pd.DatetimeIndex(frame.trade_date).is_unique
transform=AuditedTailProbabilityComposition(frame)
state={'status':'account_attempted','plan_sha256':sha(card/'plan.json'),'new_model_fits':0,'native_account_attempts':1,'bootstrap_attempts':0};atomic_json(state,card/'run_state.json')
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**account['recovery_spec']);action=Stage1OpenActionValueSpec(root=root,**account['action_spec']);execution={'initial_cash':plan['initial_cash_usd'],'tariff':account['parent']['tariff']}
path=replay_stage1_raw(frame,point_prices=account['points'],schedule=account['schedule'],plan=execution,controller=account['controller'],recovery_spec=recovery,action_spec=action,selection=account['selection'],slippage_bps=0,target_transform=transform)
assert len(path['replay'])==944 and len(path['points'])==3778 and len(transform.rows)>0
assert all(abs(x['simplex_error'])<1e-10 and abs(x['beta_error'])<1e-10 for x in transform.rows)
book=native_raw_validate(path,provider_uri=provider,point_prices=account['points'],schedule=account['schedule'],tariff=execution['tariff'],cost=0,initial_cash=execution['initial_cash'])
assert book['audit']['status']=='passed'
for role in ('Stage1_A','SOXX'):
 pd.testing.assert_frame_equal(book['authority'][['signal_date','trade_date','exit_date']].reset_index(drop=True),account['references'][role]['authority'][['signal_date','trade_date','exit_date']].reset_index(drop=True))
state.update(status='native_complete',native_account_completed=1,transform_calls=len(transform.rows));atomic_json(state,card/'run_state.json')
state['bootstrap_attempts']=1;atomic_json(state,card/'run_state.json')
result=_analyze_account(account=book['authority'],reference_soxx=account['references']['SOXX']['authority'],reference_stage1=account['references']['Stage1_A']['authority'],sampled_drawdown=-100*float(path['replay'].monitored_drawdown.max()),plan=plan)
audit=pd.DataFrame(transform.rows)
audit_summary={'transform_calls':len(audit),'open_calls':int(audit.slot.eq('open').sum()),'recovery_calls':int(audit.slot.eq('recovery').sum()),'mean_tail_probability_on_transform':float(audit.tail_probability.mean()),'mean_tail_shift':float(audit.tail_shift.mean()),'mean_composition_delta':float(audit.composition_delta.mean()),'mean_before_cash':float(audit.before_cash.mean()),'mean_after_cash':float(audit.after_cash.mean()),'max_abs_simplex_error':float(audit.simplex_error.abs().max()),'max_abs_beta_error':float(audit.beta_error.abs().max())}
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_receipt=account['receipt'],audit_summary=audit_summary)
 rec.save_objects(artifact_path='paths',candidate_0=path);rec.save_objects(artifact_path='native',candidate_0=book)
 rec.save_objects(artifact_path='analysis',SOXX_metrics=result['SOXX_metrics'],Stage1_increment=result['Stage1_increment_metrics'],annual=result['annual'],quarter=result['quarter'],rolling_summary=result['rolling_summary'],uncertainty=result['uncertainty'],gates=result['gates'],transform_audit=audit)
 rec.save_objects(artifact_path='analysis/rolling',all_windows=result['rolling'])
 rec.log_metrics(new_model_fits=0,native_accounts=1,SOXX_relative_wealth=result['SOXX_metrics']['relative_wealth'],historical_0bp_pass=int(result['historical_0bp_gate_pass']))
 rid,eid=rec.id,str(rec.experiment_id)
with R.uri_context(str(tracking)):
 rec=R.get_recorder(recorder_id=rid,experiment_id=eid)
 assert len(rec.load_object('analysis/annual'))==4 and len(rec.load_object('analysis/quarter'))==16 and len(rec.load_object('analysis/rolling/all_windows'))==819
recorder_dir=tracking/eid/rid;artifacts=recorder_dir/'artifacts';atomic_json({'recorder_id':rid,'entries':[{'path':str(x.relative_to(card)),'sha256':sha(x),'bytes':x.stat().st_size} for x in sorted(artifacts.rglob('*')) if x.is_file()]},card/'verification_manifest.json')
for kind in ('annual','quarter'):result[kind].to_csv(card/(kind+'_metrics.csv'),index=False)
summary={'experiment':plan['experiment'],'decision':'historical_0bp_pass_needs_independent_review' if result['historical_0bp_gate_pass'] else 'not_promoted_fixed_Stage1_tail_probability_composition_0bp','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'new_model_fits':0,'new_native_accounts':1,'qualified_forward_days':0,'gates':result['gates'],'SOXX_metrics':result['SOXX_metrics'],'Stage1_increment_metrics':result['Stage1_increment_metrics'],'rolling_summary':result['rolling_summary'],'uncertainty':result['uncertainty'],'transform_audit':audit_summary,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'recorder_local_dir':str(recorder_dir),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','paths/*','native/*','analysis/rolling/*']}}
atomic_json(summary,card/'summary.json')
binding=json.loads((root/'research/stage1_domain_heads_20261018/nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2026-10-09');atomic_json(binding,card/'nas_binding.json')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='stage1-tail-composition-restore-') as td:
 archive=Path(td)/'recorder.tar.gz';remote=evidence['nas_path']
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=180)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as z:names=[m.name for m in z.getmembers() if m.isfile()]
 assert any(x.endswith('/artifacts/analysis/gates') for x in names) and any(x.endswith('/artifacts/native/candidate_0') for x in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True};atomic_json(summary,card/'summary.json')
state.update(status='completed',recorder_id=rid,experiment_id=eid,historical_0bp_pass=result['historical_0bp_gate_pass']);atomic_json(state,card/'run_state.json')
metric=result['SOXX_metrics'];complete=result['annual'].loc[result['annual'].complete_period];positive=int(complete.relative_wealth.gt(1+1e-8).sum())
(card/'conclusion.md').write_text('# 原 Stage1 尾部概率条件组成：0bp 完整账户\n\n'+f"固定 Stage1 f696+b472 分配、动作、风险、恢复链；只在实际 EXECUTE 目标上以已知尾部概率执行无阈值组成变换。0 新拟合、1 本独立0bp原生Qlib账户，仍扣 IBKR Tiered 费用。SOXX 相对财富 {metric['relative_wealth']:.6f}，可用点最大回撤 {100*path['replay'].monitored_drawdown.max():.4f}%，126 日正超额窗口 {100*result['rolling_summary']['positive_fraction']:.2f}%，配对区块下界 {result['uncertainty']['daily_lower_bound']:.9f}。两个完整年度中 {positive}/{len(complete)} 跑赢。\n\n"+f"决定 `{summary['decision']}`。全部 944 日、16 季度及 819 个连续 126 日窗口保存在 Recorder，现金/股数/应收分红与 Tiered 费用经原生读回；采样风险点不等于全分钟认证，复用历史不等于独立前向。固定版本失败即关闭，不改尾部概率符号、阈值或组成公式。即使历史 0bp 通过，仍需独立复核、2/5bp 与合格前向证据。Recorder `{rid}`，NAS SHA-256 `{evidence['sha256']}`。\n")
print({'decision':summary['decision'],'relative_wealth':metric['relative_wealth'],'gates':result['gates'],'recorder_id':rid,'NAS':evidence['sha256']})

{'decision': 'not_promoted_fixed_Stage1_tail_probability_composition_0bp', 'relative_wealth': 0.5177683955206491, 'gates': {'SOXX_terminal_net_wealth': False, 'available_drawdown_30pct': False, 'CVaR5_no_worse_SOXX': True, 'complete_years_70pct': False, 'rolling_126_70pct': False, 'SOXX_paired_lower_positive': False, 'full_minute_risk_qualified': False, 'independent_forward_qualified': False}, 'recorder_id': 'ceecebbd02484ced984aafceb916bb63', 'NAS': '17ccdcb61824286195ffacb5896aa2d96372ddd14703ed9f2c6f4baf7120188d'}


## 中文结论

固定无阈值尾部概率条件组成已运行一条完整 Stage1 自有状态 0bp Tiered 账户。SOXX 相对财富 0.517768，可用点最大回撤 31.0986%，CVaR5 门通过但完整年度 0/2、全部 126 日正超额窗口 25.52%、区块下界 −0.001846112 均失败。自身路径平均变换后现金 39.81%，高于已关闭静态组成账户的 21.06%；2026Q3 相对财富 0.875，低于静态组成 0.892。这个版本决定 `not_promoted_fixed_Stage1_tail_probability_composition_0bp`，不改公式、阈值或窗口救援。全部原生账户与 NAS 收据见 `summary.json`，描述性归因见 `failure_attribution.json`；SOXX v2 目标仍未完成。